In [1]:
from bpaotu.importer import DataImporter

# careful... this will truncate everything
# but it will also refresh the schema
importer = DataImporter("~", "2025-12-17", True, False)


/env/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [ ]:
## otu.py

from sqlalchemy import (MetaData, ARRAY, Table, Column, Date, Time, Float, ForeignKey, Integer,
                        String, create_engine, select, Index, Boolean, DDL)
from sqlalchemy.sql import func
from sqlalchemy.dialects import postgresql
from sqlalchemy.ext.declarative import declarative_base, declared_attr
from sqlalchemy.orm import relationship
from sqlalchemy_utils import create_materialized_view

SCHEMA = 'otu'
Base = declarative_base(metadata=MetaData(schema=SCHEMA))


class SchemaMixin():
    """
    we use a specific schema (rather than the public schema) so that the import
    can be easily re-run, simply by deleting the schema. this also keeps
    SQLAlchemy tables out of the way of Django tables, and vice-versa
    """
    __table_args__ = {
        "schema": SCHEMA
    }


class MAG(SchemaMixin, Base):
    __tablename__ = "mags"

    # If sample_id + bin_id should be unique, we can switch to composite PK
    id = Column(Integer, primary_key=True)

    sample_id = Column(Integer, nullable=False, index=True)
    bin_id = Column(String, nullable=False, index=True)
    method = Column(String, nullable=False)

    tax = Column(String, nullable=False)
    tax_16s = Column(String, nullable=True)

    length = Column(Integer, nullable=False)
    gc_perc = Column(Float, nullable=False)
    num_contigs = Column(Integer, nullable=False)
    disparity = Column(Float, nullable=False)

    completeness = Column(Float, nullable=False)
    contamination = Column(Float, nullable=False)
    strain_het = Column(Float, nullable=False)

    coverage = Column(Float, nullable=False)
    tpm = Column(Float, nullable=False)
    quality = Column(Float, nullable=False)

    def __repr__(self):
        return (
            f"<MAG(sample_id={self.sample_id}, "
            f"bin_id={self.bin_id}, quality={self.quality})>"
        )


In [7]:
## importer.py

import csv
from pathlib import Path
from sqlalchemy import insert
import logging
import os

from sqlalchemy.orm import sessionmaker
from bpaotu.otu import (MAG, make_engine)


logger = logging.getLogger("bpaotu")

_engine = make_engine()
_session = sessionmaker(bind=_engine)()




def iter_mags_bintable_rows(path):
    with open(path, newline="") as fh:
        reader = csv.DictReader(fh)

        for r in reader:
            validate_bintable_row(r)

            yield {
                "sample_id": int(r["sample_id"]),
                "bin_id": r["Bin ID"],
                "method": r["Method"],
                "tax": r["Tax"],
                "tax_16s": r["Tax 16S"] or None,
                "length": int(r["Length"]),
                "gc_perc": float(r["GC perc"]),
                "num_contigs": int(r["Num contigs"]),
                "disparity": float(r["Disparity"]),
                "completeness": float(r["Completeness"]),
                "contamination": float(r["Contamination"]),
                "strain_het": float(r["Strain het"]),
                "coverage": float(r["Coverage"]),
                "tpm": float(r["TPM"]),
                "quality": float(r["quality"]),
            }

def validate_bintable_row(r):
    if len(r["Tax"].split(";")) > 7:    
        sample_id = r["sample_id"]
        raise DataImportError(f"Tax has more than 7 elements (delimiter=;) sample_id={sample_id} row={r}")


def load_mags_bintable(): # self
    print("Building mags bintable")
    bintable_path = "/app/data/dev/ingest/AM_mags/MAG/bins_quality_gt98.bintable"

    rows = list(iter_mags_bintable_rows(bintable_path))

    with _engine.begin() as conn: # self
        conn.execute(insert(MAG), rows)

    print("Done")

load_mags_bintable()

Building mags bintable
Tax has more than 7 elements (delimiter=;) sample_id=139804 row={'sample_id': '139804', 'Bin ID': 'maxbin.020.fasta.contigs', 'Method': 'DAS', 'Tax': 'k_Bacteria;n_FCB group;n_Bacteroidetes/Chlorobi group;p_Bacteroidetes;c_Flavobacteriia;o_Flavobacteriales;f_Flavobacteriaceae;g_Robiginitalea;s_Robiginitalea biformata', 'Tax 16S': 'superkingdom:Bacteria;phylum:Proteobacteria;class:Alphaproteobacteria', 'Length': '3413813', 'GC perc': '55.61', 'Num contigs': '32', 'Disparity': '0.596', 'Completeness': '99.35', 'Contamination': '0.16', 'Strain het': '0.00', 'Coverage': '17.499', 'TPM': '6124.810', 'quality': '98.55'}


TypeError: exceptions must derive from BaseException